Importing libraries and loading dataset

In [30]:
import pandas as pd
import numpy as np
import plotly.express as px
from scipy.stats import chi2_contingency

In [34]:
file_path = r"C:\Users\Ecem\Desktop\TIL\1. TIL6022 - Python\Project\TIL6022-Group-Project\Data\Processed\merged_eurostat_clean_V2.csv"

df = pd.read_csv(file_path)

Preparing data for the analysis

In [39]:
# Select relevant columns
base_features = [
    "geo", "TIME_PERIOD",
    "Consignment_full_train_load_THS_T",
    "Consignment_full_wagon_load_THS_T",
    "Consignment_total_THS_T",
    "NST_TOTAL_MIO_TKM"
]

# Automatically detect all NST commodity columns (GT01–GT20)
nst_columns = [col for col in df.columns if col.startswith("NST_GT")]
print(f"Detected {len(nst_columns)} NST commodity columns.")

# ------------------------------------------------------------
# 1. Convert to long format
# ------------------------------------------------------------
df_long = df.melt(
    id_vars=base_features,
    value_vars=nst_columns,
    var_name="NST2007",
    value_name="Volume_MIO_TKM"
)

# Clean NST2007 labels
df_long["NST2007"] = (
    df_long["NST2007"]
    .str.replace("NST_GT", "", regex=False)
    .str.replace("_MIO_TKM", "", regex=False)
)

# ------------------------------------------------------------
# 3. Add NST2007 code-to-description mapping (Eurostat standard)
# ------------------------------------------------------------
nst_labels = {
    "01": "Agriculture and forestry products",
    "02": "Coal and crude petroleum",
    "03": "Metal ores and quarrying products",
    "04": "Food, beverages, tobacco",
    "05": "Textiles and leather products",
    "06": "Wood and cork products",
    "07": "Paper and printed matter",
    "08": "Refined petroleum products",
    "09": "Chemicals and man-made fibres",
    "10": "Rubber and plastic products",
    "11": "Non-metallic mineral products",
    "12": "Basic and fabricated metals",
    "13": "Machinery and equipment",
    "14": "Electrical machinery",
    "15": "Transport equipment",
    "16": "Furniture and other goods",
    "17": "Secondary raw materials and waste",
    "18": "Grouped goods (mixed consignments)",
    "19": "Unidentifiable goods",
    "20": "Empty packaging and return loads"
}

df_long["NST2007_desc"] = df_long["NST2007"].map(nst_labels)

# Drop missing or zero freight volumes
df_long = df_long.dropna(subset=["Volume_MIO_TKM"]).copy()
df_long = df_long[df_long["Volume_MIO_TKM"] > 0].copy()

print("Long-format data ready:", df_long.shape)

Detected 20 NST commodity columns.
Long-format data ready: (6337, 9)
